# DSCI 552 – Homework 3

**Name:** Pooja Nandyappa Subashchandra  
**GitHub Username:** nandyapp-afk  
**USC ID:** 1768721225

In [1]:
import os
import re

import numpy as np
import pandas as pd

# Relative path (forward slashes only): data/AReM/<activity>/dataset<k>.csv
DATA_DIR = "data/AReM"

ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
COLS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
SERIES = COLS[1:]          # the 6 time series
N_BOOT = 1000              # bootstrap resamples
ALPHA = 0.10               # 90% confidence interval
SEED = 0

## Problem 1 – Time Series Classification, Part 1: Feature Creation/Extraction

### 1(a)–(b) Load the data and split into train / test

Some files need minor cleaning, so `load_instance` skips lines starting with `#`, accepts comma or whitespace delimiters, and drops malformed rows.
**Test set:** datasets 1 and 2 of `bending1` and `bending2`, and datasets 1, 2, 3 of every other folder. All other datasets are **train**.

In [2]:
def load_instance(path):
    """Read one AReM file into a DataFrame with the 7 columns in COLS."""
    rows = []
    with open(path, errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p for p in re.split(r"[,\s]+", line) if p]
            if len(parts) != len(COLS):
                continue
            try:
                rows.append([float(p) for p in parts])
            except ValueError:
                continue
    return pd.DataFrame(rows, columns=COLS)


def is_test(activity, k):
    if activity in ("bending1", "bending2"):
        return k in (1, 2)
    return k in (1, 2, 3)


raw = {}                                       # (activity, dataset number) -> DataFrame
for act in ACTIVITIES:
    folder = f"{DATA_DIR}/{act}"
    files = [f for f in os.listdir(folder) if re.match(r"dataset\d+\.csv$", f)]
    files.sort(key=lambda f: int(re.findall(r"\d+", f)[0]))
    for f in files:
        k = int(re.findall(r"\d+", f)[0])
        raw[(act, k)] = load_instance(f"{folder}/{f}")

n_test = sum(is_test(a, k) for (a, k) in raw)
print(f"Instances: {len(raw)}  (train = {len(raw) - n_test}, test = {n_test})")
print("Files without exactly 480 rows:", {key: len(df) for key, df in raw.items() if len(df) != 480} or "none")

Instances: 88  (train = 69, test = 19)
Files without exactly 480 rows: {('sitting', 8): 479}


### 1(c)i Common time-domain features

- **Location:** minimum, maximum, mean, median, first and third quartile
- **Spread:** standard deviation, variance, range, interquartile range
- **Shape:** skewness, kurtosis
- **Magnitude:** root mean square, signal energy, mean absolute value
- **Dynamics:** mean-crossing rate, number of peaks, mean absolute change, linear trend (slope), autocorrelation

### 1(c)ii Feature extraction

For each of the 6 time series of every instance: minimum, maximum, mean, median, standard deviation, 1st quartile and 3rd quartile (7 × 6 = 42 features). The suffix is the time series number (1 = avg_rss12, 2 = var_rss12, 3 = avg_rss13, 4 = var_rss13, 5 = avg_rss23, 6 = var_rss23).

In [3]:
FEATURES = {
    "min": np.min,
    "max": np.max,
    "mean": np.mean,
    "median": np.median,
    "std": lambda x: np.std(x, ddof=1),
    "1st_quart": lambda x: np.percentile(x, 25),
    "3rd_quart": lambda x: np.percentile(x, 75),
}

rows = []
for (act, k), df in raw.items():
    rec = {"activity": act, "dataset": k,
           "split": "test" if is_test(act, k) else "train"}
    for i, s in enumerate(SERIES, start=1):
        x = df[s].to_numpy()
        for name, fn in FEATURES.items():
            rec[f"{name}{i}"] = fn(x)
    rows.append(rec)

features = pd.DataFrame(rows)
features.insert(0, "instance", np.arange(1, len(features) + 1))
feat_cols = [c for c in features.columns if c not in ("instance", "activity", "dataset", "split")]

print("Feature table shape:", features.shape, "->", len(feat_cols), "features")
features.head()

Feature table shape: (88, 46) -> 42 features


,instance,activity,dataset,split,min1,max1,mean1,median1,std1,1st_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


### 1(c)iii Standard deviation of each feature and 90% bootstrap confidence intervals

For each feature I compute its standard deviation across the 88 instances. The 90% bootstrap confidence interval comes from resampling the 88 instances with replacement 1000 times, recomputing the standard deviation on each resample, and taking the 5th and 95th percentiles.

In [4]:
def bootstrap_std_ci(X, n_boot=N_BOOT, alpha=ALPHA, seed=SEED):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_std = X[idx].std(axis=1, ddof=1)               # (n_boot, n_features)
    lo = np.percentile(boot_std, 100 * alpha / 2, axis=0)
    hi = np.percentile(boot_std, 100 * (1 - alpha / 2), axis=0)
    return X.std(axis=0, ddof=1), lo, hi


s, lo, hi = bootstrap_std_ci(features[feat_cols].to_numpy())
ci = pd.DataFrame({"feature": feat_cols, "std": s, "ci90_low": lo, "ci90_high": hi})
pd.set_option("display.max_rows", 100)
ci.round(4)

,feature,std,ci90_low,ci90_high
0,min1,9.5700,8.2955,10.8195
1,max1,4.3944,3.3354,5.2734
2,mean1,5.3357,4.7062,5.8512
3,median1,5.4401,4.8407,6.0001
4,std1,1.7722,1.5732,1.9567
5,1st_quart1,6.1536,5.6151,6.6414
6,3rd_quart1,5.1389,4.3359,5.8408
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6257,5.3754
9,mean2,1.5742,1.3942,1.7021


### 1(c)iv Three most important time-domain features

**Selected features: minimum, mean and maximum.** The mean gives the typical level of each series, and the minimum and maximum give the range it moves through, so together they summarize each series' level and spread with simple, robust statistics.

## Problem 2 – ISLR 3.7.4

**(a)** The **cubic** regression has the lower (or equal) training RSS. Linear is a special case of cubic (β₂ = β₃ = 0), so the extra terms can only reduce training RSS, even if they just fit noise.

**(b)** The **linear** regression has the lower test RSS. The cubic model overfits: it adds variance but no reduction in bias, since the true relationship is already linear.

**(c)** The **cubic** regression has the lower training RSS, for the same reason as in (a).

**(d)** There is **not enough information** to tell. If the true relationship is closer to cubic than to linear, the cubic fit has the lower test RSS; if it is closer to linear, the linear fit has the lower test RSS.